In [3]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("data/raw/world_bank_indicators.csv")

df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Year range: {df['year'].min()}–{df['year'].max()}")

df.head()

Rows: 1,848
Columns: 6
Year range: 1960–2025


,country,country_code,year,indicator,indicator_code,value
0,Ghana,GHA,1960,GDP (current US$),NY.GDP.MKTP.CD,1.217230e+09
1,Ghana,GHA,1961,GDP (current US$),NY.GDP.MKTP.CD,1.302674e+09
2,Ghana,GHA,1962,GDP (current US$),NY.GDP.MKTP.CD,1.382516e+09
3,Ghana,GHA,1963,GDP (current US$),NY.GDP.MKTP.CD,1.540798e+09
4,Ghana,GHA,1964,GDP (current US$),NY.GDP.MKTP.CD,1.731296e+09


In [4]:
nigeria = df[df["country"] == "Nigeria"].copy()

summary_rows = []

for indicator in nigeria["indicator"].unique():
    temp = (
        nigeria[
            (nigeria["indicator"] == indicator)
            & (nigeria["value"].notna())
        ]
        .sort_values("year")
    )

    first = temp.iloc[0]
    latest = temp.iloc[-1]

    summary_rows.append({
        "indicator": indicator,
        "first_year": first["year"],
        "first_value": first["value"],
        "latest_year": latest["year"],
        "latest_value": latest["value"],
        "absolute_change": latest["value"] - first["value"],
        "percent_change": (
            (latest["value"] - first["value"])
            / first["value"]
        ) * 100
        if first["value"] != 0
        else None,
    })

nigeria_summary = pd.DataFrame(summary_rows)

nigeria_summary

,indicator,first_year,first_value,latest_year,latest_value,absolute_change,percent_change
0,GDP (current US$),1960,4.196175e+09,2025,2.907944e+11,2.865982e+11,6829.987336
1,GDP per capita (current US$),1960,9.313701e+01,2025,1.224254e+03,1.131117e+03,1214.465769
2,Inflation,1960,5.444327e+00,2025,2.301012e+01,1.756580e+01,322.644012
3,Internet users,1990,0.000000e+00,2024,4.121140e+01,4.121140e+01,NaN
4,Life expectancy,1960,3.720700e+01,2024,5.463500e+01,1.742800e+01,46.840648
5,Population,1960,4.505378e+07,2025,2.375278e+08,1.924740e+08,427.209418
6,Unemployment,1991,3.831000e+00,2025,3.064000e+00,-7.670000e-01,-20.020882


In [5]:
turning_points = []

for indicator in nigeria["indicator"].unique():
    temp = nigeria[
        (nigeria["indicator"] == indicator)
        & (nigeria["value"].notna())
    ].copy()

    min_row = temp.loc[temp["value"].idxmin()]
    max_row = temp.loc[temp["value"].idxmax()]

    turning_points.append({
        "indicator": indicator,
        "min_year": int(min_row["year"]),
        "min_value": min_row["value"],
        "max_year": int(max_row["year"]),
        "max_value": max_row["value"],
    })

turning_points = pd.DataFrame(turning_points)

turning_points

,indicator,min_year,min_value,max_year,max_value
0,GDP (current US$),1960,4.196175e+09,2019,6.682200e+11
1,GDP per capita (current US$),1960,9.313701e+01,2019,3.189813e+03
2,Inflation,1967,-3.726337e+00,1995,7.283550e+01
3,Internet users,1990,0.000000e+00,2024,4.121140e+01
4,Life expectancy,1960,3.720700e+01,2024,5.463500e+01
5,Population,1960,4.505378e+07,2025,2.375278e+08
6,Unemployment,2024,3.045000e+00,2020,5.714000e+00


In [6]:
benchmark_rows = []

for indicator in df["indicator"].unique():

    for country in df["country"].unique():

        temp = df[
            (df["country"] == country)
            & (df["indicator"] == indicator)
            & (df["value"].notna())
        ].sort_values("year")

        if temp.empty:
            continue

        latest = temp.iloc[-1]

        benchmark_rows.append({
            "country": country,
            "indicator": indicator,
            "latest_year": int(latest["year"]),
            "latest_value": latest["value"],
        })

latest_benchmark = pd.DataFrame(benchmark_rows)

benchmark_pivot = latest_benchmark.pivot(
    index="indicator",
    columns="country",
    values="latest_value",
)

benchmark_pivot

country,Ghana,Kenya,Nigeria,South Africa
indicator,,,,
GDP (current US$),1.142099e+11,1.359413e+11,2.907944e+11,4.271843e+11
GDP per capita (current US$),3.257159e+03,2.362861e+03,1.224254e+03,6.597715e+03
Inflation,1.420468e+01,4.069129e+00,2.301012e+01,3.205505e+00
Internet users,7.217570e+01,3.497623e+01,4.121140e+01,7.835760e+01
Life expectancy,6.569500e+01,6.383400e+01,5.463500e+01,6.631200e+01
Population,3.506427e+07,5.753249e+07,2.375278e+08,6.474732e+07
Unemployment,2.979000e+00,5.449000e+00,3.064000e+00,3.239100e+01


In [7]:
comparison_rows = []

for indicator in df["indicator"].unique():

    temp = df[
        (df["indicator"] == indicator)
        & (df["value"].notna())
    ].copy()

    country_latest_years = (
        temp.groupby("country")["year"].max()
    )

    common_year = country_latest_years.min()

    comparison = temp[
        temp["year"] == common_year
    ][["country", "indicator", "year", "value"]]

    comparison_rows.append(comparison)

comparable_benchmark = pd.concat(
    comparison_rows,
    ignore_index=True
)

comparable_pivot = comparable_benchmark.pivot(
    index=["indicator", "year"],
    columns="country",
    values="value"
)

comparable_pivot

,country,Ghana,Kenya,Nigeria,South Africa
indicator,year,,,,
GDP (current US$),2025,1.142099e+11,1.359413e+11,2.907944e+11,4.271843e+11
GDP per capita (current US$),2025,3.257159e+03,2.362861e+03,1.224254e+03,6.597715e+03
Inflation,2025,1.420468e+01,4.069129e+00,2.301012e+01,3.205505e+00
Internet users,2024,7.217570e+01,3.497623e+01,4.121140e+01,7.835760e+01
Life expectancy,2024,6.569500e+01,6.383400e+01,5.463500e+01,6.631200e+01
Population,2025,3.506427e+07,5.753249e+07,2.375278e+08,6.474732e+07
Unemployment,2025,2.979000e+00,5.449000e+00,3.064000e+00,3.239100e+01


In [8]:
powerbi_df = df.copy()

powerbi_df["value"] = pd.to_numeric(
    powerbi_df["value"],
    errors="coerce"
)

powerbi_df["is_nigeria"] = (
    powerbi_df["country"] == "Nigeria"
)

powerbi_df["period"] = pd.cut(
    powerbi_df["year"],
    bins=[1959, 1969, 1979, 1989, 1999, 2009, 2019, 2029],
    labels=[
        "1960s",
        "1970s",
        "1980s",
        "1990s",
        "2000s",
        "2010s",
        "2020s",
    ]
)

OUTPUT_PATH = Path(
    "data/processed/nigeria_at_66_powerbi.csv"
)

powerbi_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(powerbi_df):,}")
print(f"Columns: {powerbi_df.shape[1]}")

powerbi_df.head()

Saved: data\processed\nigeria_at_66_powerbi.csv
Rows: 1,848
Columns: 8


,country,country_code,year,indicator,indicator_code,value,is_nigeria,period
0,Ghana,GHA,1960,GDP (current US$),NY.GDP.MKTP.CD,1.217230e+09,False,1960s
1,Ghana,GHA,1961,GDP (current US$),NY.GDP.MKTP.CD,1.302674e+09,False,1960s
2,Ghana,GHA,1962,GDP (current US$),NY.GDP.MKTP.CD,1.382516e+09,False,1960s
3,Ghana,GHA,1963,GDP (current US$),NY.GDP.MKTP.CD,1.540798e+09,False,1960s
4,Ghana,GHA,1964,GDP (current US$),NY.GDP.MKTP.CD,1.731296e+09,False,1960s


In [2]:
from pathlib import Path

print("Working directory:")
print(Path.cwd())

Working directory:
c:\Users\abola\OneDrive\Documents\my project 2026 data analytics\Nigeria-at-66
